# Reproducibility and research artifacts

Reproducibility has several identities: scientific configuration, source data, random streams, software environment, and exported results. MFDRO records the parts it controls and exposes the remaining boundaries rather than combining them into one opaque file.

This notebook verifies deterministic reruns, configuration serialization, portable result persistence, timezone preservation, and checksum failure after controlled tampering.

In [ ]:
import hashlib
import json
import platform
from importlib.metadata import version
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import mfdro
from mfdro import (
    ConfigurationError,
    DataContractError,
    FrequencySpec,
    MultiFrequencySignal,
    SignalConfig,
    SignalPath,
)

plt.style.use("seaborn-v0_8-whitegrid")
INK = "#183b4e"
ACCENT = "#008c82"

## 1. Give the source panel its own identity

`SignalConfig.digest` does not identify data. The example therefore computes a separate checksum of a canonical CSV representation. A production archive should hash the actual governed source artifact and its transformation manifest.

In [ ]:
rng = np.random.default_rng(404)
dates = pd.bdate_range("2020-01-02", "2024-12-31")
factor = rng.normal(0.0001, 0.007, size=(len(dates), 1))
returns = pd.DataFrame(
    factor + rng.normal(0.0, 0.004, size=(len(dates), 5)),
    index=dates,
    columns=[f"asset_{index}" for index in range(5)],
)
source_payload = returns.to_csv(date_format="%Y-%m-%d", float_format="%.17g")
source_digest = hashlib.sha256(source_payload.encode("utf-8")).hexdigest()

pd.Series({"rows": len(returns), "assets": returns.shape[1], "source_sha256": source_digest})

## 2. Serialize the complete scientific configuration

The configuration is immutable, strict, and schema-versioned. Unknown fields fail rather than being ignored. Its digest changes whenever a serialized scientific setting changes.

In [ ]:
FREQUENCIES = (
    FrequencySpec("daily", 1.0),
    FrequencySpec("weekly", 5.0, rule="W-FRI", min_observations=3),
    FrequencySpec("monthly", 21.0, rule="ME", min_observations=10),
)
config = SignalConfig.projected(
    frequency_specs=FREQUENCIES,
    n_projections=80,
    n_quantiles=96,
    random_state=20250301,
).with_updates(
    frequency_weighting="explicit",
    explicit_frequency_weights=(3.0, 2.0, 1.0),
    barycenter_weights=(1.0, 1.0, 1.0),
)
round_trip = SignalConfig.from_json(config.to_json())
changed = config.with_updates(n_projections=config.n_projections + 1)

assert round_trip == config
assert round_trip.digest == config.digest
assert changed.digest != config.digest
pd.Series({"config_digest": config.digest, "changed_digest": changed.digest})

In [ ]:
invalid_payload = config.to_dict()
invalid_payload["unreviewed_setting"] = True
try:
    SignalConfig.from_dict(invalid_payload)
except ConfigurationError as error:
    strict_error = str(error)
else:
    raise AssertionError("An unknown configuration field should have failed.")

strict_error

## 3. Repeat the same numerical experiment

Path seeds are derived from the base random state, namespace, and formation month. Identical inputs therefore produce identical estimate, audit, and skipped tables.

In [ ]:
engine = MultiFrequencySignal(config)
path_a = engine.estimate_path(
    returns,
    lookback_months=24,
    seed_namespace="archive_demo",
)
path_b = engine.estimate_path(
    returns.copy(),
    lookback_months=24,
    seed_namespace="archive_demo",
)

pd.testing.assert_frame_equal(path_a.estimates, path_b.estimates)
pd.testing.assert_frame_equal(path_a.audit, path_b.audit)
pd.testing.assert_frame_equal(path_a.skipped, path_b.skipped)
path_a.estimates[["date", "rho", "seed", "config_digest"]].tail()

## 4. Configuration identity is not data identity

The next path uses the same configuration and seeds but perturbs one historical month. The configuration digest remains constant while affected signal values change.

In [ ]:
altered_returns = returns.copy()
altered_returns.loc["2024-06", "asset_0"] += 0.01
altered_path = engine.estimate_path(
    altered_returns,
    lookback_months=24,
    seed_namespace="archive_demo",
)
comparison = pd.concat(
    [path_a.sqrt_rho.rename("original source"), altered_path.sqrt_rho.rename("altered source")],
    axis=1,
)
assert altered_path.config.digest == path_a.config.digest
assert not comparison["original source"].equals(comparison["altered source"])

axis = comparison.plot(figsize=(9, 3.5), color=[INK, ACCENT], linewidth=1.5)
axis.set(
    title="Same configuration, different source artifact",
    xlabel="formation date",
    ylabel=r"$\sqrt{\rho}$",
)
plt.tight_layout()

## 5. Save, load, and verify a portable result bundle

`SignalPath.save` writes JSON tables, the exact configuration, and a manifest. Loading checks file hashes, schemas, aligned dates, and configuration identity before returning a result.

In [ ]:
with TemporaryDirectory() as temporary_directory:
    root = Path(temporary_directory)
    config_path = config.write_json(root / "standalone-config.json")
    destination = path_a.save(root / "signal-path")
    restored_config = SignalConfig.read_json(config_path)
    restored_path = SignalPath.load(destination)
    manifest = json.loads((destination / "manifest.json").read_text(encoding="utf-8"))
    artifact_files = sorted(path.name for path in destination.iterdir())
    observed_hash = hashlib.sha256((destination / "estimates.json").read_bytes()).hexdigest()

    assert restored_config == config
    pd.testing.assert_frame_equal(restored_path.estimates, path_a.estimates)
    assert observed_hash == manifest["sha256"]["estimates.json"]

artifact_summary = pd.Series(
    {
        "files": ", ".join(artifact_files),
        "format_version": manifest["format_version"],
        "package_version": manifest["package_version"],
        "estimate_rows": manifest["rows"]["estimates"],
        "config_digest": manifest["config_digest"],
    }
)
artifact_summary

## 6. Detect accidental alteration

The demonstration appends one byte to a copied result table. The numerical content may appear unchanged to a casual reader, but the stored checksum must fail.

In [ ]:
with TemporaryDirectory() as temporary_directory:
    destination = path_a.save(Path(temporary_directory) / "signal-path")
    estimate_file = destination / "estimates.json"
    estimate_file.write_text(estimate_file.read_text(encoding="utf-8") + " ", encoding="utf-8")
    try:
        SignalPath.load(destination)
    except DataContractError as error:
        integrity_error = str(error)
    else:
        raise AssertionError("A modified artifact should have failed its checksum.")

integrity_error

## 7. Preserve timezone-aware formation dates

Timezone information is part of the temporal contract and survives the portable JSON-table round trip.

In [ ]:
zurich_returns = returns.tz_localize("Europe/Zurich")
last_date = zurich_returns.index[-1]
timezone_path = engine.estimate_path(
    zurich_returns,
    lookback_months=24,
    formation_dates=[last_date],
    reference_calendar=zurich_returns.index,
    on_insufficient="raise",
)
with TemporaryDirectory() as temporary_directory:
    restored_timezone_path = SignalPath.load(
        timezone_path.save(Path(temporary_directory) / "timezone-path")
    )

assert str(restored_timezone_path.estimates.loc[0, "date"].tzinfo) == "Europe/Zurich"
restored_timezone_path.estimates[["date", "rho"]]

## 8. Record the numerical environment

Checksums establish file integrity, not cross-version numerical equivalence. An archive should retain Python and dependency versions in addition to the package manifest.

In [ ]:
environment = pd.Series(
    {
        "python": platform.python_version(),
        "mfdro": mfdro.__version__,
        "numpy": version("numpy"),
        "pandas": version("pandas"),
        "POT": version("POT"),
        "scikit-learn": version("scikit-learn"),
    },
    name="version",
)
environment

## Reproducibility boundary

The saved MFDRO path is a verifiable calculation artifact. Full empirical reconstruction additionally requires the exact source snapshot, its licence and transformation code, the point-in-time membership ledger, the authoritative calendar, and the downstream policy that turns dispersion into a decision. None of those identities should be inferred from `config.digest`.